# Advanced Transformations

The largest exam domain (29%). We cover advanced transformations in PySpark and SQL: window functions (ranking, lag/lead), array operations (explode), JSON processing, CTEs, subqueries, CASE WHEN, UDFs, and higher-order functions. Each technique is demonstrated in both PySpark and SQL.

| Exam Domain | Weight |
|---|---|
| ELT with Spark SQL and Python | 29% |

> **BONUS MODULE** — This content is not required for the DEA exam but enriches Day 2 with SQL transformation techniques.

---

## Setup

Initialize the environment, import required PySpark functions, and configure catalog/schema references for this module.

---

In [0]:
%run ../../setup/00_setup

### Configuration

Import all required PySpark modules (window, functions, types) and set the active catalog and schema.

In [0]:
from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, row_number, rank, dense_rank, lag, lead,
    sum as _sum, avg, count, max as _max, min as _min,
    to_date, date_trunc, date_add, add_months, last_day,
    explode, posexplode, sequence, from_json, to_json, schema_of_json,
    current_timestamp, round as _round, lit, when, coalesce,
    udf, array, transform, filter, exists
)
from pyspark.sql.types import *
import datetime

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {BRONZE_SCHEMA}")

## Part A: PySpark Transformations

Deep dive into PySpark's advanced transformation capabilities — window functions (ranking, lag/lead, rolling aggregations), complex types (arrays with explode/posexplode), JSON parsing, and date/time functions for temporal analysis.

---

### Window Functions — Basics

**Building a window spec:**
```python
Window.partitionBy("col")          # define partition groups
      .orderBy(col("col").desc())  # sort within partition
      .rowsBetween(start, end)     # window frame (optional)
```

| Element | Signature | Description |
|---|---|---|
| `partitionBy()` | `Window.partitionBy(col, ...)` | Splits data into groups (like GROUP BY) |
| `orderBy()` | `.orderBy(col)` | Sorts rows within each partition |
| `rowsBetween()` | `.rowsBetween(start, end)` | Frame based on physical row offset |
| `rangeBetween()` | `.rangeBetween(start, end)` | Frame based on column value range (requires `orderBy`) |

**Use cases:** Ranking, time comparisons (lag/lead), moving aggregations, trend analysis

In [0]:
# Prepare sample order data
orders_data = [
    (1, 1, "2024-01-15", 150.0),
    (2, 2, "2024-01-16", 200.0),
    (3, 1, "2024-02-10", 300.0),
    (4, 3, "2024-02-12", 100.0),
    (5, 2, "2024-03-05", 450.0),
    (6, 1, "2024-03-15", 250.0),
    (7, 3, "2024-03-20", 180.0),
    (8, 2, "2024-04-01", 320.0),
    (9, 1, "2024-04-10", 400.0),
    (10, 3, "2024-04-15", 220.0),
]

In [0]:
orders_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("order_date", StringType(), False),
    StructField("amount", DoubleType(), False)
])

In [0]:
orders_df = spark.createDataFrame(orders_data, orders_schema)
orders_df = orders_df.withColumn("order_date", to_date(col("order_date")))
orders_df.display()

### Demo: Ranking — row_number, rank, dense_rank

| Function | Signature | Description |
|---|---|---|
| `row_number()` | `row_number().over(window_spec)` | Unique row number within partition — no ties (1, 2, 3, 4, ...) |
| `rank()` | `rank().over(window_spec)` | Rank with gaps on ties (1, 2, 2, **4**, ...) |
| `dense_rank()` | `dense_rank().over(window_spec)` | Rank without gaps on ties (1, 2, 2, **3**, ...) |

In [0]:
# Ranking orders for each customer by amount (descending)
window_spec = Window.partitionBy("customer_id").orderBy(col("amount").desc())

orders_ranked = orders_df.withColumn("row_num", row_number().over(window_spec)) \
    .withColumn("rank", rank().over(window_spec)) \
    .withColumn("dense_rank", dense_rank().over(window_spec))

In [0]:
orders_ranked.orderBy("customer_id", "row_num").display()

### Demo: Lag and Lead Functions

| Function | Signature | Description |
|---|---|---|
| `lag()` | `lag(col, offset=1, default=None).over(window_spec)` | Value from *offset* rows before; `None` when no prior row exists |
| `lead()` | `lead(col, offset=1, default=None).over(window_spec)` | Value from *offset* rows ahead; `None` when no following row exists |

In [0]:
# Comparison with previous and next order
window_spec_time = Window.partitionBy("customer_id").orderBy("order_date")

orders_lag_lead = orders_df \
    .withColumn("prev_amount", lag("amount", 1).over(window_spec_time)) \
    .withColumn("next_amount", lead("amount", 1).over(window_spec_time)) \
    .withColumn("amount_change", col("amount") - col("prev_amount")) \
    .withColumn("amount_change_pct", 
                _round((col("amount") - col("prev_amount")) / col("prev_amount") * 100, 2))

In [0]:
orders_lag_lead.orderBy("customer_id", "order_date").display()

### Rolling Windows — Moving Aggregations

| Element | Signature / Constant | Description |
|---|---|---|
| `rowsBetween()` | `rowsBetween(start, end)` | Frame based on physical row count |
| `rangeBetween()` | `rangeBetween(start, end)` | Frame based on value of the `orderBy` column |
| `Window.unboundedPreceding` | `-sys.maxsize` | Start of partition |
| `Window.unboundedFollowing` | `sys.maxsize` | End of partition |
| `Window.currentRow` | `0` | Current row |

**Frame examples:**
- `rowsBetween(Window.unboundedPreceding, Window.currentRow)` → cumulative sum
- `rowsBetween(-2, 0)` → 3-row sliding window (2 previous + current)
- `rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)` → entire partition

In [0]:
# Rolling sum (all previous + current)
window_cumulative = Window.partitionBy("customer_id") \
    .orderBy("order_date") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

orders_cumulative = orders_df \
    .withColumn("cumulative_sum", _sum("amount").over(window_cumulative)) \
    .withColumn("cumulative_count", count("order_id").over(window_cumulative)) \
    .withColumn("cumulative_avg", _round(avg("amount").over(window_cumulative), 2))

In [0]:
orders_cumulative.orderBy("customer_id", "order_date").display()

In [0]:
# Moving average - 3 last orders
window_moving_3 = Window.partitionBy("customer_id") \
    .orderBy("order_date") \
    .rowsBetween(-2, 0)  # 2 previous + current = 3 rows

orders_moving_avg = orders_df \
    .withColumn("moving_avg_3", _round(avg("amount").over(window_moving_3), 2)) \
    .withColumn("moving_sum_3", _sum("amount").over(window_moving_3)) \
    .withColumn("moving_max_3", _max("amount").over(window_moving_3)) \
    .withColumn("moving_min_3", _min("amount").over(window_moving_3))

### Calculating Moving Average

Applying moving window to calculate:
- **moving_avg_3**: Average of last 3 orders
- **moving_sum_3**: Sum of last 3 orders
- **moving_max_3**: Max of last 3 orders
- **moving_min_3**: Min of last 3 orders

**rowsBetween(-2, 0)** means: 2 previous rows + current = 3-row window

In [0]:
orders_moving_avg.orderBy("customer_id", "order_date").display()

## Deduplication

Remove duplicate rows using `dropDuplicates()` (Python) or `DISTINCT` (SQL).

| Method | Syntax | Notes |
|--------|--------|-------|
| `dropDuplicates()` | `df.dropDuplicates()` | All columns |
| `dropDuplicates(cols)` | `df.dropDuplicates(["id", "date"])` | Subset of columns |
| SQL DISTINCT | `SELECT DISTINCT ...` | SQL equivalent |
| `QUALIFY` | `ROW_NUMBER() = 1` | Keep first per group (window-based dedup) |

> **Exam Tip:** `dropDuplicates(["customer_id", "order_date"])` deduplicates based on a subset of columns — useful for SCD Type 1 patterns.

In [ ]:
from pyspark.sql.functions import row_number
from pyspark.sql import Window

# Sample data with duplicates
dup_data = [(1, 'Alice', '2024-01-15', 'email'), (1, 'Alice', '2024-01-15', 'web'),
            (2, 'Bob', '2024-01-16', 'email'), (3, 'Carol', '2024-01-15', 'email')]
dup_df = spark.createDataFrame(dup_data, ['customer_id', 'name', 'date', 'channel'])

# dropDuplicates on subset of columns (keep first occurrence by customer_id + date)
dedup_df = dup_df.dropDuplicates(['customer_id', 'date'])
print(f'Before: {dup_df.count()} rows | After: {dedup_df.count()} rows')
dedup_df.show()

# Window-based dedup: keep lowest channel alphabetically per customer+date
w = Window.partitionBy('customer_id', 'date').orderBy('channel')
dedup_window = dup_df.withColumn('rn', row_number().over(w)).filter('rn = 1').drop('rn')
dedup_window.show()

## Part B: SQL Transformations

Explore the same transformation patterns using Spark SQL — including temp views, JOINs, window functions, CTEs, subqueries (scalar, correlated, EXISTS/IN), CASE WHEN expressions, and DDL operations for persisting results.

---

### Spark SQL Basics

`spark.sql()` executes SQL and returns a DataFrame. Supports all standard SQL, uses **Catalyst Optimizer**.

In [0]:
# Example: Simple SQL query
result = spark.sql("""
    SELECT 
        'Hello Spark SQL' as message,
        current_date() as today,
        current_timestamp() as now
""")
display(result)

### Creating Test Data

Preparing data for Spark SQL demonstration:

In [0]:
# Orders data
orders_data = [
    (1, 101, "2024-01-15", 250.00, "completed"),
    (2, 102, "2024-01-16", 150.00, "completed"),
    (3, 101, "2024-01-20", 320.00, "completed"),
    (4, 103, "2024-02-01", 180.00, "pending"),
    (5, 101, "2024-02-10", 420.00, "completed"),
    (6, 102, "2024-02-15", 90.00, "cancelled"),
    (7, 103, "2024-03-01", 550.00, "completed"),
    (8, 104, "2024-03-05", 280.00, "completed"),
    (9, 101, "2024-03-10", 175.00, "completed"),
    (10, 102, "2024-03-15", 340.00, "completed"),
]

orders_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("order_date", StringType(), False),
    StructField("amount", DoubleType(), False),
    StructField("status", StringType(), False)
])

orders_df = spark.createDataFrame(orders_data, orders_schema) \
    .withColumn("order_date", F.to_date("order_date"))

In [0]:
# Customers data
customers_data = [
    (101, "Jan", "Kowalski", "Premium", "Warszawa"),
    (102, "Anna", "Nowak", "Standard", "Krakow"),
    (103, "Piotr", "Wisniewski", "Premium", "Gdansk"),
    (104, "Maria", "Wojcik", "Standard", "Poznan"),
]

customers_schema = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("first_name", StringType(), False),
    StructField("last_name", StringType(), False),
    StructField("tier", StringType(), False),
    StructField("city", StringType(), False)
])

customers_df = spark.createDataFrame(customers_data, customers_schema)
display(orders_df.limit(5))
display(customers_df)

### Registering Temp Views

To use DataFrames in SQL queries, they must be registered as temporary views.

**View types:**
- `createOrReplaceTempView()` - local view for the session
- `createOrReplaceGlobalTempView()` - global view (accessible via `global_temp.name`)

In [0]:
# Registering temporary views
orders_df.createOrReplaceTempView("orders")
customers_df.createOrReplaceTempView("customers")

In [0]:
# Now we can use SQL
spark.sql("SELECT * FROM orders LIMIT 5").display()

### SQL vs DataFrame API Comparison

Side-by-side comparison showing that DataFrame API and Spark SQL produce identical results and execution plans.

**Task:** Find total value of completed orders per customer

In [0]:
# DataFrame API Approach
result_df = orders_df \
    .filter(F.col("status") == "completed") \
    .groupBy("customer_id") \
    .agg(
        F.count("*").alias("orders_count"),
        F.sum("amount").alias("total_amount"),
        F.round(F.avg("amount"), 2).alias("avg_amount")
    ) \
    .orderBy(F.col("total_amount").desc())
display(result_df)

In [0]:
# Spark SQL Approach
result_sql = spark.sql("""
    SELECT 
        customer_id,
        COUNT(*) as orders_count,
        SUM(amount) as total_amount,
        ROUND(AVG(amount), 2) as avg_amount
    FROM orders
    WHERE status = 'completed'
    GROUP BY customer_id
    ORDER BY total_amount DESC
""")
display(result_sql)

**Comparison:** Both approaches yield identical results and execution plans.

### JOIN — Combining DataFrames

**String concatenation functions used in joins:**

| Function | Signature | Description |
|---|---|---|
| `concat_ws()` | `concat_ws(sep, *cols)` | Joins column values with a separator — ignores NULLs |
| `concat()` | `concat(*cols)` | Concatenates without separator — returns NULL if any column is NULL |

In [0]:
# DataFrame API - JOIN
joined_df = orders_df \
    .join(customers_df, "customer_id", "inner") \
    .select(
        "order_id",
        F.concat_ws(" ", "first_name", "last_name").alias("customer_name"),
        "tier",
        "order_date",
        "amount",
        "status"
    )
display(joined_df.limit(5))

In [0]:
# Spark SQL - JOIN
joined_sql = spark.sql("""
    SELECT 
        o.order_id,
        CONCAT_WS(' ', c.first_name, c.last_name) as customer_name,
        c.tier,
        o.order_date,
        o.amount,
        o.status
    FROM orders o
    INNER JOIN customers c ON o.customer_id = c.customer_id
""")
display(joined_sql.limit(5))

### Window Functions in SQL

| Type | Signature | Description |
|---|---|---|
| Ranking | `ROW_NUMBER() OVER (PARTITION BY ... ORDER BY ...)` | Unique sequential number within the partition |
| Analytic | `SUM(col) OVER (PARTITION BY ... ORDER BY ... ROWS BETWEEN ...)` | Rolling aggregation over a window frame |

**Key clauses:**
- `PARTITION BY` — defines the grouping (window scope)
- `ORDER BY` — determines sort order within the partition
- `ROWS BETWEEN` — defines the window frame (e.g., `UNBOUNDED PRECEDING AND CURRENT ROW`)

In [0]:
# Order ranking per customer
ranking_sql = spark.sql("""
    SELECT 
        order_id,
        customer_id,
        order_date,
        amount,
        ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date) as order_sequence,
        RANK() OVER (PARTITION BY customer_id ORDER BY amount DESC) as amount_rank,
        DENSE_RANK() OVER (PARTITION BY customer_id ORDER BY amount DESC) as amount_dense_rank
    FROM orders
    WHERE status = 'completed'
    ORDER BY customer_id, order_date
""")
display(ranking_sql)

### LAG and LEAD - Change Analysis

Using `LAG()` and `LEAD()` in SQL to compare each order with the previous/next one and calculate amount differences.

In [0]:
# Comparison with previous order
lag_lead_sql = spark.sql("""
    SELECT 
        order_id,
        customer_id,
        order_date,
        amount,
        LAG(amount, 1) OVER (PARTITION BY customer_id ORDER BY order_date) as prev_amount,
        LEAD(amount, 1) OVER (PARTITION BY customer_id ORDER BY order_date) as next_amount,
        amount - LAG(amount, 1) OVER (PARTITION BY customer_id ORDER BY order_date) as amount_change
    FROM orders
    WHERE status = 'completed'
    ORDER BY customer_id, order_date
""")
display(lag_lead_sql)

### Running Totals and Moving Averages

Cumulative sums and sliding-window averages using SQL `ROWS BETWEEN` frame specifications.

In [0]:
# Running total and moving average
running_sql = spark.sql("""
    SELECT 
        order_id,
        customer_id,
        order_date,
        amount,
        SUM(amount) OVER (
            PARTITION BY customer_id 
            ORDER BY order_date 
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) as cumulative_amount,
        ROUND(AVG(amount) OVER (
            PARTITION BY customer_id 
            ORDER BY order_date 
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ), 2) as moving_avg_3
    FROM orders
    WHERE status = 'completed'
    ORDER BY customer_id, order_date
""")
display(running_sql)

### CTE (Common Table Expressions)

`WITH` — named reusable subqueries; more readable and easier to debug than nested subqueries.

```sql
WITH cte_name AS (
    SELECT ...           -- CTE definition
),
cte_name2 AS (           -- multiple CTEs can be chained
    SELECT ... FROM cte_name
)
SELECT * FROM cte_name2  -- using the CTE
```

| Feature | CTE (`WITH`) | Subquery |
|---|---|---|
| Reusability | yes | no |
| Readability | high | low when nested |
| Recursion | since Spark 3.4 / DBR 14.2+ | no |
| Performance | identical (Catalyst) | identical |

> **Note:** Recursive CTEs are supported since Spark 3.4 / DBR 14.2+.

In [0]:
# CTE - Customer Analysis
cte_analysis = spark.sql("""
    WITH customer_orders AS (
        SELECT 
            customer_id,
            COUNT(*) as orders_count,
            SUM(amount) as total_spent,
            AVG(amount) as avg_order_value
        FROM orders
        WHERE status = 'completed'
        GROUP BY customer_id
    ),
    customer_ranking AS (
        SELECT 
            *,
            RANK() OVER (ORDER BY total_spent DESC) as spending_rank,
            CASE 
                WHEN total_spent >= 500 THEN 'High Value'
                WHEN total_spent >= 300 THEN 'Medium Value'
                ELSE 'Low Value'
            END as value_segment
        FROM customer_orders
    )
    SELECT 
        cr.*,
        c.first_name,
        c.last_name,
        c.tier,
        c.city
    FROM customer_ranking cr
    JOIN customers c ON cr.customer_id = c.customer_id
    ORDER BY spending_rank
""")
display(cte_analysis)

### Reusing CTEs

A single CTE can be referenced multiple times in the same query, avoiding redundant subqueries.

In [0]:
# CTE used multiple times
multi_cte = spark.sql("""
    WITH monthly_stats AS (
        SELECT 
            DATE_TRUNC('month', order_date) as month,
            customer_id,
            SUM(amount) as monthly_spent
        FROM orders
        WHERE status = 'completed'
        GROUP BY DATE_TRUNC('month', order_date), customer_id
    )
    SELECT 
        month,
        COUNT(DISTINCT customer_id) as active_customers,
        SUM(monthly_spent) as total_revenue,
        ROUND(AVG(monthly_spent), 2) as avg_customer_spend,
        MAX(monthly_spent) as max_customer_spend
    FROM monthly_stats
    GROUP BY month
    ORDER BY month
""")
display(multi_cte)

### CASE WHEN and Advanced Expressions

**SQL:**
```sql
-- Searched CASE (arbitrary conditions)
CASE WHEN condition1 THEN result1
     WHEN condition2 THEN result2
     ELSE default_result
END

-- Simple CASE (comparison to a value)
CASE expression
     WHEN value1 THEN result1
     WHEN value2 THEN result2
     ELSE default_result
END
```

**PySpark:** `when(condition, value).when(condition2, value2).otherwise(default)`

Conditional logic in SQL for order segmentation, status mapping, and safe value handling.

In [0]:
# Order segmentation
case_when_sql = spark.sql("""
    SELECT 
        order_id,
        customer_id,
        amount,
        CASE 
            WHEN amount >= 500 THEN 'Large'
            WHEN amount >= 200 THEN 'Medium'
            ELSE 'Small'
        END as order_size,
        CASE status
            WHEN 'completed' THEN 1
            WHEN 'pending' THEN 0
            ELSE -1
        END as status_code,
        COALESCE(amount, 0) as amount_safe
    FROM orders
    ORDER BY amount DESC
""")
display(case_when_sql)

### NULLIF, COALESCE, NVL

| Function | Signature | Description |
|---|---|---|
| `COALESCE` | `COALESCE(col1, col2, ...)` | Returns the first non-NULL value from the argument list |
| `NULLIF` | `NULLIF(expr1, expr2)` | Returns `NULL` when `expr1 = expr2`; otherwise returns `expr1` |
| `NVL` | `NVL(expr, default)` | Alias for `COALESCE(expr, default)` — replaces NULL with a default value |
| `IFNULL` | `IFNULL(expr, default)` | Alias for `NVL` (Spark SQL / MySQL compat.) |

**PySpark:** `coalesce(col1, col2, ...)` — identical behaviour to SQL `COALESCE`

In [0]:
# Handling NULLs
null_handling = spark.sql("""
    SELECT 
        order_id,
        amount,
        status,
        NULLIF(status, 'cancelled') as status_or_null,
        COALESCE(NULLIF(status, 'cancelled'), 'N/A') as status_clean,
        NVL(amount, 0) as amount_nvl
    FROM orders
""")
display(null_handling)

## Part C: User-Defined Functions (UDFs)

| Type | Language | Registration | Performance |
|---|---|---|---|
| Python UDF | Python | `@udf` decorator / `spark.udf.register()` | Slowest (serialization) |
| Pandas UDF | Python | `@pandas_udf` | Fast (Arrow) |
| SQL UDF | SQL | `CREATE FUNCTION` | Fast (no serialization) |

> **Exam Note:** Python UDFs = slow (row-by-row serialization). SQL UDFs = faster (Spark engine).

### Example: Python UDF

In [0]:
# Python UDF - classify order value
@udf("string")
def classify_order(amount):
    if amount is None:
        return "Unknown"
    elif amount > 500:
        return "High Value"
    elif amount > 100:
        return "Medium Value"
    else:
        return "Low Value"

# Apply UDF to DataFrame (using temp view registered in Part B)
df_orders = spark.table("orders")
df_classified = df_orders.withColumn("order_class", classify_order(col("amount")))
display(df_classified.select("order_id", "amount", "order_class").limit(10))

### Example: SQL UDF

In [0]:
%sql
-- SQL UDF - no serialization overhead
CREATE OR REPLACE FUNCTION classify_order_sql(amount DOUBLE)
RETURNS STRING
RETURN CASE
    WHEN amount IS NULL THEN 'Unknown'
    WHEN amount > 500 THEN 'High Value'
    WHEN amount > 100 THEN 'Medium Value'
    ELSE 'Low Value'
END;

In [0]:
# Use the SQL UDF (using temp view registered in Part B)
df_udf = spark.sql("""SELECT order_id, amount, classify_order_sql(amount) AS order_class
FROM orders
LIMIT 10;""")

display(df_udf)

## Part D: Higher-Order Functions

Operate on arrays **without** unpacking them (`explode`):

| Function | SQL Signature | PySpark Signature | Description |
|---|---|---|---|
| `TRANSFORM` | `TRANSFORM(arr, x -> expr)` | `transform(col, f: lambda x)` | Applies function `f` to each element; returns a new array of the same length |
| `FILTER` | `FILTER(arr, x -> condition)` | `filter(col, f: lambda x)` | Keeps only elements satisfying the condition |
| `EXISTS` | `EXISTS(arr, x -> condition)` | `exists(col, f: lambda x)` | Returns `true` if **at least one** element satisfies the condition |
| `FORALL` | `FORALL(arr, x -> condition)` | `forall(col, f: lambda x)` | Returns `true` if **all** elements satisfy the condition |
| `AGGREGATE` | `AGGREGATE(arr, start, (acc, x) -> merge [, finish])` | `aggregate(col, zero, merge, finish)` | Reduces an array to a single value (fold-left) |

> **Exam Note:** Higher-order functions > EXPLODE + GROUP BY for array operations.

### Example: TRANSFORM and FILTER in SQL

In [0]:
%sql
-- Create sample data with arrays
CREATE OR REPLACE TEMP VIEW order_items AS
SELECT 1 AS order_id, array(10.0, 25.5, 8.0, 120.0) AS item_prices
UNION ALL
SELECT 2, array(5.0, 15.0, 200.0, 50.0)
UNION ALL
SELECT 3, array(99.0, 1.0, 45.0);

In [0]:
%sql 
select * from order_items

In [0]:
%sql
-- TRANSFORM: apply 10% discount to all items
-- FILTER: keep only items above 20
-- EXISTS: check if any item is above 100
SELECT
    order_id,
    item_prices,
    TRANSFORM(item_prices, x -> ROUND(x * 0.9, 2)) AS discounted_prices,
    FILTER(item_prices, x -> x > 20) AS expensive_items,
    EXISTS(item_prices, x -> x > 100) AS has_premium_item
FROM order_items;

### Example: Higher-Order Functions in PySpark

In [0]:
from pyspark.sql.functions import transform, filter, exists, aggregate

df_items = spark.sql("SELECT * FROM order_items")

result = df_items.select(
    "order_id",
    "item_prices",
    transform("item_prices", lambda x: x * 0.9).alias("discounted"),
    filter("item_prices", lambda x: x > 20).alias("expensive"),
    exists("item_prices", lambda x: x > 100).alias("has_premium")
)
display(result)

## Summary

| Category | PySpark | SQL |
|---|---|---|
| Window Functions | `Window.partitionBy().orderBy()` | `OVER (PARTITION BY ... ORDER BY ...)` |
| Ranking | `row_number()`, `dense_rank()` | `ROW_NUMBER()`, `DENSE_RANK()` |
| Lag/Lead | `lag()`, `lead()` | `LAG()`, `LEAD()` |
| Complex Types | `explode()`, `posexplode()` | `EXPLODE()`, `LATERAL VIEW` |
| JSON | `from_json()`, `to_json()` | `from_json()`, `to_json()` |
| Date Functions | `date_trunc()`, `date_add()` | `DATE_TRUNC()`, `DATE_ADD()` |
| CTE | DataFrame chain | `WITH ... AS (...)` |
| Subqueries | `.filter(col.isin(...))` | `WHERE x IN (SELECT ...)` |
| Conditional | `when().otherwise()` | `CASE WHEN ... END` |
| UDFs | `@udf` decorator | `CREATE FUNCTION` |
| Higher-Order | `transform()`, `filter()` | `TRANSFORM()`, `FILTER()` |

> **Exam Tip:** DataFrame API = SQL performance (Catalyst Optimizer). Python UDFs are slower than SQL UDFs due to serialization. Higher-order functions preferred over EXPLODE for arrays. `DENSE_RANK` has no gaps, `RANK` has gaps, `ROW_NUMBER` is unique.

---

> **← M05: Incremental Processing | Day 2 | M07: Medallion & Lakeflow →**

## Cleanup

Drop temporary tables, views, and UDFs created during this module. Uncomment the lines below to execute.

---

In [0]:
# Optional cleanup
# Uncomment below to execute cleanup:

# spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{GOLD_SCHEMA}.customer_summary")
# spark.sql("DROP FUNCTION IF EXISTS classify_order_sql")
# spark.catalog.clearCache()
# print("Cleanup completed.")